# 03 — ECB Bronze to Silver

## 1. Read June Exchange Rates

Read the raw ECB exchange rate data for June 2025.

In [ ]:
df_ecb_june = (
    spark.read
    .option("header", True)
    .csv("Files/bronze/ecb/fx_usd_2025-06-01_2025-06-30_raw.csv")
)

df_ecb_june.printSchema()
df_ecb_june.show(10)

## 2. Transform June Exchange Rates

Convert the date column and cast the exchange rate to double.

In [ ]:
from pyspark.sql.functions import to_date

df_ecb_silver = df_ecb_june.select(
    to_date("TIME_PERIOD").alias("rate_date"),
    "OBS_VALUE")
df_ecb_silver.printSchema()
df_ecb_silver.show(5)

In [ ]:
df_ecb_silver = df_ecb_silver.select(
    "rate_date",
    df_ecb_silver.OBS_VALUE.cast("double").alias("usd_per_eur"))
df_ecb_silver.printSchema()
df_ecb_silver.show(5)

## 3. Read July Exchange Rates

Read the raw ECB exchange rate data for July 2025.

In [ ]:
df_ecb_july = (
    spark.read
    .option("header", True)
    .csv("Files/bronze/ecb/fx_usd_2025-07-01_2025-07-31_raw.csv")
)

df_ecb_july.printSchema()
df_ecb_july.show()

## 4. Transform July Exchange Rates

Convert TIME_PERIOD to date and OBS_VALUE to double.

In [ ]:
df_ecb_july_silver = df_ecb_july.select(
    to_date("TIME_PERIOD").alias("rate_date"),
    df_ecb_july.OBS_VALUE.cast("double").alias("usd_per_eur")
)

df_ecb_july_silver.printSchema()
df_ecb_july_silver.show(5)
print("July rows:", df_ecb_july_silver.count())

## 5. Combine June and July Exchange Rates

Combine both monthly datasets into a single DataFrame.

In [ ]:
df_ecb_combined = df_ecb_silver.unionByName(df_ecb_july_silver)

print("Combined rows:", df_ecb_combined.count())
df_ecb_combined.orderBy("rate_date").show(5)

## 6. Data Quality Checks

Check duplicate dates, missing dates, and invalid exchange rates.

In [ ]:
unique_dates = df_ecb_combined.dropDuplicates(["rate_date"]).count()

total_rows = df_ecb_combined.count()

duplicates = total_rows - unique_dates

print("Unique dates:", unique_dates)
print("Total rows:", total_rows)
print("Duplicates:", duplicates)

In [ ]:
null_dates = df_ecb_combined.filter(df_ecb_combined.rate_date.isNull()).count()
print("Null dates:", null_dates)

In [ ]:
invalid_rates = df_ecb_combined.filter((df_ecb_combined.usd_per_eur.isNull()) | 
(df_ecb_combined.usd_per_eur <= 0)).count()
print(invalid_rates)

## 7. Write Silver Delta Table

Save the cleaned ECB exchange rates as a Delta table in the Lakehouse.

In [ ]:
df_ecb_combined.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("silver_ecb_fx")


## 8. Validate Silver Output

Verify the saved table, row count, column count, and sample records.

In [ ]:
df_check = spark.table("silver_ecb_fx")
print("Rows:", df_check.count())
print("Columns:", len(df_check.columns))

df_check.orderBy("rate_date").show(5)